**Questions**
1. The LOB df already contains transaction data, while the FUT df doesn't.
    - In the paper it says that transaction data for FUT has been used, but it is only added as a col to indicate transacted volume + no. of transactions in the LOB data
    - So, how is the FUT transaction data used?
3. Transaction is indicated by TradeNo>0?
4. Absolute Error vs MSE
5. no-change benchmark vs prior-day-mean benchmark
6. What lagged horizons to use for prediction? Two options:
    - Infer from Cestonaro & Trimpe
    - Test on 2 stocks, one first three months, the other last three months 

**My understanding of how these dfs where constructed (tbc)**
1. LOB df dictates sampling frequency and 'buckets' into which trades are sorted -> multiple trades can go into one bucket
2. Trades are sorted according to the rule ts(LOB) <= ts(Trade)

**My understanding of how to proceed with these two dfs (also tbc)**
1. Load data and construct morning session, 08:17 to 11:45 UTC, and afternoon session, 12:17 to 16:15 UTC
    - LOB
    - FUT
2. Format data, only keep
    - LOB: Timestamp (datetime & int), Security ID, L1-QImb, Microprice -- check for value of other imb levels
    - FUT: Timestamp (datetime & int), Security ID, Microprice -- check for value of imb in FUT 
3. Build new feature-target datasets
    - Features: Non-overlapping L1-QImb & Microprice at different lags
    - Targets: Microprice log-return at different forecast horizons
4. Combine FUT and LOB data into one df
    - Data points are matched according to the rule ts(LOB) <= ts(FUT)

In [1]:
import pandas as pd
import numpy as np
import os
import time
from tqdm.auto import tqdm

import importlib
import utils.data_preprocessing as du

importlib.reload(du)

<module 'utils.data_preprocessing' from '/home/jovyan/_shared_storage/temp_stud/MA_Otto/utils/data_preprocessing.py'>

In [2]:
data_folder = 'Price_Measures_2025-01-28'

In [4]:
overall_start = time.time()

# ------------------------------------------------------------------
# Preload and preprocess all FUT data once
# ------------------------------------------------------------------
fut_cache = {}

for day in tqdm(du.SAMPLE_DATES, desc="Loading FUT data"):

    fut = pd.read_csv(f"{data_folder}/FUT_DAX Futures/FUT_DAX Futures_{day}.csv.gz")
    fut = fut[['Timestamp', 'MicroPrice_tick-based_10_1s']]

    fut = du.compute_feature_target_matrix(
        df=fut[['Timestamp', 'MicroPrice_tick-based_10_1s']],
        ts_col='Timestamp',
        target_cols=[],
        feature_cols=['MicroPrice_tick-based_10_1s'],
        resample_flag=False,
        horizons=['-5m', '-2.5m', '-1m', '-30s', '15s', '10s', '-2s', '-1s', '-100ms']
    )

    fut_cache[day] = fut

# ------------------------------------------------------------------
# Process LOB files
# ------------------------------------------------------------------
for day in tqdm(du.SAMPLE_DATES, desc="Processing days"):

    day_start = time.time()

    # Retrieve preprocessed FUT data
    fut = fut_cache[day]

    for symbol in tqdm(du.SYMBOLS[:10], desc=f"Symbols ({day})", leave=False):

        lob = pd.read_csv(f"{data_folder}/{symbol}/{symbol}_{day}.csv.gz")

        lob = du.filter_trading_hours(lob)
        lob['TransactionPrice'] = du.compute_transaction_price(lob)

        lob = lob[['Timestamp_Europe/Berlin', 'Timestamp', 'RelativeSpread', 'L1-QImb'] + du.PRICE_MEASURES]

        lob = du.compute_feature_target_matrix(
            df=lob,
            ts_col='Timestamp',
            target_cols=du.PRICE_MEASURES,
            feature_cols=['L1-QImb', 'MicroPrice_tick-based_10_1s'],
            resample_flag=True,
            horizons=['-5m', '-2.5m', '-1m', '-30s', '15s', '10s', '-2s', '-1s', '-100ms',
                      '100ms', '1s', '2s', '10s', '15s', '30s', '1m', '2.5m', '5m']
        )

        merged = pd.merge_asof(
            lob,
            fut,
            on="Timestamp",
            direction="backward",
            suffixes=("_LOB", "_FUT")
        )

        merged = merged.drop(columns=['TransactionPrice', 'MidPriceQW', 'MidPriceCQW', 'MicroPrice_tick-based_10_1s_FUT', 'MicroPrice_tick-based_10_1s_LOB'])

        merged.to_parquet(f'Feature_Targets/{symbol}_{day}.parquet')

    day_end = time.time()
    tqdm.write(f"[{day}] processing completed in {day_end - day_start:.2f} seconds")

overall_end = time.time()
print(f"Overall processing completed in {overall_end - overall_start:.2f} seconds")

Loading FUT data:   0%|          | 0/127 [00:00<?, ?it/s]

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [3]:
# open & pre-process daily fut data, open & pre-process daily lob data, combine the two
overall_start = time.time()

for day in tqdm(du.SAMPLE_DATES, desc="Processing days"):

    day_start = time.time()
    
    fut = pd.read_csv(f"{data_folder}/FUT_DAX Futures/FUT_DAX Futures_{day}.csv.gz")
    fut = fut[['Timestamp', 'MicroPrice_tick-based_10_1s']] # Check if imbalance provides sign add predictability

    fut = du.compute_feature_target_matrix(
             df=fut,
             ts_col = 'Timestamp',
             target_cols = [],
             feature_cols = ['MicroPrice_tick-based_10_1s'],
             resample_flag = False,
             horizons = ['-5m', '-2.5m', '-1m', '-30s', '15s', '10s', '-2s', '-1s', '-100ms'])

    for symbol in tqdm(du.SYMBOLS[:10], desc=f"Symbols ({day})", leave=False):

        lob = pd.read_csv(f"{data_folder}/{symbol}/{symbol}_{day}.csv.gz")
        lob = du.filter_trading_hours(lob)

        lob['TransactionPrice'] = du.compute_transaction_price(lob)
        lob = lob[['Timestamp_Europe/Berlin', 'Timestamp', 'RelativeSpread'] + ['L1-QImb'] + du.PRICE_MEASURES]


        lob = du.compute_feature_target_matrix(
                 df=lob,
                 ts_col = 'Timestamp',
                 target_cols = du.PRICE_MEASURES,
                 feature_cols = ['L1-QImb', 'MicroPrice_tick-based_10_1s'],
                 resample_flag = True,
                 horizons = ['-5m', '-2.5m', '-1m', '-30s', '15s', '10s', '-2s', '-1s', '-100ms',
                         '100ms', '1s', '2s', '10s', '15s', '30s', '1m', '2.5m', '5m'])

        merged = pd.merge_asof(
            lob,
            fut,
            on="Timestamp",
            direction="backward", #Chooses the nearest earlier-or-equal timestamp in LOB for every entry in FUT
            suffixes=("_LOB", "_FUT")
        )

        merged = merged.drop(columns=['TransactionPrice', 'MidPriceQW', 'MidPriceCQW', 'MicroPrice_tick-based_10_1s_FUT', 'MicroPrice_tick-based_10_1s_LOB'])
        merged.to_parquet(f'Feature_Targets/{symbol}_{day}.parquet')
    
    day_end = time.time()
    tqdm.write(f"[{day}] processing completed in {day_end - day_start:.2f} seconds")


overall_end = time.time()
print(f"Overall processing completed in {overall_end - overall_start:.2f} seconds")


Processing days:   0%|          | 0/127 [00:00<?, ?it/s]

[2023-01-02] FUT processing completed in 4.70 seconds


Symbols (2023-01-02):   0%|          | 0/10 [00:00<?, ?it/s]

  CS_Deutsche Post completed in 3.39 seconds
  CS_Henkel completed in 0.76 seconds
  CS_Fresenius completed in 0.88 seconds
  CS_Siemens completed in 3.41 seconds
  CS_Beiersdorf completed in 0.49 seconds
  CS_RWE completed in 1.22 seconds
  CS_Muenchener Rueckversicherungs-Gesellschaft completed in 1.23 seconds
  CS_Continental completed in 1.70 seconds
  CS_Mercedes Benz completed in 3.25 seconds
  CS_Zalando completed in 2.33 seconds
[2023-01-03] FUT processing completed in 21.55 seconds


Symbols (2023-01-03):   0%|          | 0/10 [00:00<?, ?it/s]

...

  CS_Deutsche Post completed in 3.78 seconds
  CS_Henkel completed in 1.07 seconds
  CS_Fresenius completed in 1.19 seconds
  CS_Siemens completed in 6.99 seconds
  CS_Beiersdorf completed in 0.83 seconds
  CS_RWE completed in 1.55 seconds
  CS_Muenchener Rueckversicherungs-Gesellschaft completed in 1.91 seconds
  CS_Continental completed in 1.66 seconds
  CS_Mercedes Benz completed in 4.65 seconds
  CS_Zalando completed in 2.73 seconds
[2023-06-29] FUT processing completed in 13.37 seconds


Symbols (2023-06-29):   0%|          | 0/10 [00:00<?, ?it/s]

  CS_Deutsche Post completed in 4.12 seconds
  CS_Henkel completed in 1.02 seconds
  CS_Fresenius completed in 0.98 seconds
  CS_Siemens completed in 6.20 seconds
  CS_Beiersdorf completed in 0.75 seconds
  CS_RWE completed in 1.72 seconds
  CS_Muenchener Rueckversicherungs-Gesellschaft completed in 1.51 seconds
  CS_Continental completed in 1.67 seconds
  CS_Mercedes Benz completed in 5.91 seconds
  CS_Zalando completed in 1.66 seconds
[2023-06-30] FUT processing completed in 14.92 seconds


Symbols (2023-06-30):   0%|          | 0/10 [00:00<?, ?it/s]

  CS_Deutsche Post completed in 4.13 seconds
  CS_Henkel completed in 1.04 seconds
  CS_Fresenius completed in 0.96 seconds
  CS_Siemens completed in 4.78 seconds
  CS_Beiersdorf completed in 0.73 seconds
  CS_RWE completed in 1.36 seconds
  CS_Muenchener Rueckversicherungs-Gesellschaft completed in 1.91 seconds
  CS_Continental completed in 1.39 seconds
  CS_Mercedes Benz completed in 5.06 seconds
  CS_Zalando completed in 1.44 seconds
Overall processing completed in 6382.86 seconds


In [4]:
res = pd.read_parquet("Feature_Targets/CS_Deutsche Post_2023-01-02.parquet")
res["L1-QImb"].min()

0.000712

In [5]:
# add something to make sure that no column is only NaN

In [6]:
res.columns

Index(['bin', 'Timestamp_Europe/Berlin', 'SecurityID', 'Timestamp',
       'RelativeSpread', 'L1-QImb', 'MidPrice',
       'T_TransactionPrice_LogReturn_15s', 'T_TransactionPrice_LogReturn_10s',
       'T_TransactionPrice_LogReturn_100ms', 'T_TransactionPrice_LogReturn_1s',
       'T_TransactionPrice_LogReturn_2s', 'T_TransactionPrice_LogReturn_30s',
       'T_TransactionPrice_LogReturn_1m', 'T_TransactionPrice_LogReturn_2.5m',
       'T_TransactionPrice_LogReturn_5m', 'T_MidPrice_LogReturn_15s',
       'T_MidPrice_LogReturn_10s', 'T_MidPrice_LogReturn_100ms',
       'T_MidPrice_LogReturn_1s', 'T_MidPrice_LogReturn_2s',
       'T_MidPrice_LogReturn_30s', 'T_MidPrice_LogReturn_1m',
       'T_MidPrice_LogReturn_2.5m', 'T_MidPrice_LogReturn_5m',
       'T_MidPriceQW_LogReturn_15s', 'T_MidPriceQW_LogReturn_10s',
       'T_MidPriceQW_LogReturn_100ms', 'T_MidPriceQW_LogReturn_1s',
       'T_MidPriceQW_LogReturn_2s', 'T_MidPriceQW_LogReturn_30s',
       'T_MidPriceQW_LogReturn_1m', 'T_MidPric

In [7]:
lob = pd.read_csv(f"{data_folder}/CS_Deutsche Post/CS_Deutsche Post_{day}.csv.gz")
lob.columns

Index(['Timestamp_Europe/Berlin', 'MarketID', 'MarketSegmentID', 'SecurityID',
       'Timestamp', 'TransactTime', 'L1-BidPrice', 'L1-BidSize',
       'L1-BidNoOrder', 'L1-AskPrice', 'L1-AskSize', 'L1-AskNoOrder',
       'L2-BidPrice', 'L2-BidSize', 'L2-BidNoOrder', 'L2-AskPrice',
       'L2-AskSize', 'L2-AskNoOrder', 'L3-BidPrice', 'L3-BidSize',
       'L3-BidNoOrder', 'L3-AskPrice', 'L3-AskSize', 'L3-AskNoOrder',
       'L4-BidPrice', 'L4-BidSize', 'L4-BidNoOrder', 'L4-AskPrice',
       'L4-AskSize', 'L4-AskNoOrder', 'L5-BidPrice', 'L5-BidSize',
       'L5-BidNoOrder', 'L5-AskPrice', 'L5-AskSize', 'L5-AskNoOrder',
       'L6-BidPrice', 'L6-BidSize', 'L6-BidNoOrder', 'L6-AskPrice',
       'L6-AskSize', 'L6-AskNoOrder', 'L7-BidPrice', 'L7-BidSize',
       'L7-BidNoOrder', 'L7-AskPrice', 'L7-AskSize', 'L7-AskNoOrder',
       'L8-BidPrice', 'L8-BidSize', 'L8-BidNoOrder', 'L8-AskPrice',
       'L8-AskSize', 'L8-AskNoOrder', 'L9-BidPrice', 'L9-BidSize',
       'L9-BidNoOrder', 'L9-AskPrice

In [8]:
fut1 = pd.read_csv(f"{data_folder}/FUT_DAX Futures/FUT_DAX Futures_2023-01-02.csv.gz")
fut2 = pd.read_csv(f"{data_folder}/FUT_DAX Futures/FUT_DAX Futures_2023-01-03.csv.gz")

print(len(fut1), len(fut2))

988290 4334478


In [9]:
988.290, 4.334.478

SyntaxError: invalid syntax (3550513145.py, line 1)